# 初始化价格模型（首次运行）

命名为 bronze_prices、silver_dim_*、silver_fact_*。
只创建缺失表，引用数据只插入缺失键；不会覆盖手工维护的门店名称和坐标。
门店信息来自项目现有代码，尚未进行外部核实；未知值留空。
标准商品来自 map.xlsx，生产商品牌留空。
日常运行 ingest_daily_prices；本 Notebook 不需要每天运行。


In [ ]:
statements = ["CREATE TABLE IF NOT EXISTS workspace.default.silver_dim_brand (\nbrand_id STRING NOT NULL PRIMARY KEY,\nbrand_name STRING NOT NULL, country_code STRING\n) USING DELTA COMMENT '超市连锁品牌；不是商品生产商'", "CREATE TABLE IF NOT EXISTS workspace.default.silver_dim_category (\ncategory_id INT NOT NULL PRIMARY KEY,\ncategory_name STRING NOT NULL, category_name_zh STRING\n) USING DELTA COMMENT '商品分类：蛋、奶、肉'", "CREATE TABLE IF NOT EXISTS workspace.default.silver_dim_store (\nstore_key STRING NOT NULL PRIMARY KEY,\nsource STRING NOT NULL, store_id STRING NOT NULL,\nbrand_id STRING NOT NULL REFERENCES workspace.default.silver_dim_brand(brand_id),\nstore_name STRING, address STRING, city STRING, region STRING, country_code STRING,\nlatitude DECIMAL(10,7), longitude DECIMAL(10,7), location_verified BOOLEAN,\nlocation_source_url STRING, coordinate_source_url STRING, location_checked_date DATE,\nlocation_status STRING, is_christchurch_urban BOOLEAN\n) USING DELTA COMMENT '门店键为 source:store_id；未核实的名称、地址和坐标留空'", "CREATE TABLE IF NOT EXISTS workspace.default.silver_dim_product (\nproduct_id STRING NOT NULL PRIMARY KEY,\nproduct_name STRING NOT NULL,\ncategory_id INT REFERENCES workspace.default.silver_dim_category(category_id),\nspecification STRING, quantity DECIMAL(12,4), quantity_unit STRING,\ncomparison_unit STRING, product_brand_name STRING,\nis_analysis_selected BOOLEAN,\nanalysis_rank INT,\nselection_notes STRING,\ncomparison_ready BOOLEAN\n) USING DELTA COMMENT '标准商品；沿用 map.xlsx 的四位商品 ID；商品生产商品牌待维护'", "CREATE TABLE IF NOT EXISTS workspace.default.silver_dim_product_alias (\nsource STRING NOT NULL, original_name STRING NOT NULL,\noriginal_specification STRING NOT NULL,\nproduct_id STRING NOT NULL REFERENCES workspace.default.silver_dim_product(product_id),\nPRIMARY KEY (source, original_name, original_specification)\n) USING DELTA COMMENT '三家超市原始品名及规格到标准商品的映射；精确匹配'", "CREATE TABLE IF NOT EXISTS workspace.default.silver_dim_date (\ndate_id INT NOT NULL PRIMARY KEY, calendar_date DATE NOT NULL,\nyear INT, quarter INT, month INT, day INT, day_of_week INT, is_weekend BOOLEAN\n) USING DELTA COMMENT '日期维度；day_of_week 为星期一=1 至星期日=7'", "CREATE TABLE IF NOT EXISTS workspace.default.silver_fact_product_price (\nobservation_id STRING NOT NULL PRIMARY KEY,\ndate_id INT NOT NULL REFERENCES workspace.default.silver_dim_date(date_id),\ncollected_date DATE NOT NULL,\nstore_key STRING NOT NULL REFERENCES workspace.default.silver_dim_store(store_key),\nproduct_id STRING REFERENCES workspace.default.silver_dim_product(product_id),\ncategory_id INT REFERENCES workspace.default.silver_dim_category(category_id),\nsource STRING NOT NULL, store_id STRING NOT NULL,\noriginal_name STRING NOT NULL, original_specification STRING,\ntotal_price DECIMAL(12,4), reported_unit_price DECIMAL(12,4), reported_unit STRING,\nis_promo BOOLEAN, product_mapping_status STRING, price_quality_status STRING,\nsource_file STRING, ingested_at TIMESTAMP,\nsource_product_id STRING,\nproduct_url STRING,\nmember_price DECIMAL(12,4),\nmember_unit_price DECIMAL(12,4),\nwas_price DECIMAL(12,4),\nstore_verification_status STRING,\nraw_specification STRING,\nraw_reported_unit STRING,\nspecification_quality_status STRING,\nunit_quality_status STRING,\nnormalized_unit STRING,\nnormalized_unit_price DECIMAL(16,6),\nnormalized_member_unit_price DECIMAL(16,6)\n) USING DELTA COMMENT '每日门店商品报价；原始品名和规格区分未映射商品；reported 单价未标准化'", "CREATE TABLE IF NOT EXISTS workspace.default.silver_dim_product_source (\n source STRING NOT NULL, source_product_id STRING NOT NULL,\n product_id STRING REFERENCES workspace.default.silver_dim_product(product_id),\n original_name STRING, original_specification STRING, category_id INT,\n product_url STRING, mapping_status STRING, mapping_note STRING,\n PRIMARY KEY (source,source_product_id)\n) USING DELTA COMMENT 'Source SKU registry. Only approved/exact_reference mappings resolve a product; pending stays unmapped.' ", "MERGE INTO workspace.default.silver_dim_brand t USING (SELECT * FROM VALUES ('woolworths', 'Woolworths', 'NZ'),\n('newworld', 'New World', 'NZ'),\n('paknsave', 'PAK\\'nSAVE', 'NZ') AS v(brand_id, brand_name, country_code)) s ON t.brand_id=s.brand_id WHEN NOT MATCHED THEN INSERT (brand_id, brand_name, country_code) VALUES (s.brand_id, s.brand_name, s.country_code)", "MERGE INTO workspace.default.silver_dim_category t USING (SELECT * FROM VALUES ('1', 'Egg', '蛋类'),\n('2', 'Milk', '奶类'),\n('3', 'Meat', '肉类') AS v(category_id, category_name, category_name_zh)) s ON t.category_id=s.category_id WHEN NOT MATCHED THEN INSERT (category_id, category_name, category_name_zh) VALUES (s.category_id, s.category_name, s.category_name_zh)", "MERGE INTO workspace.default.silver_dim_store t USING (SELECT * FROM VALUES ('woolworths:001', 'woolworths', '001', 'woolworths', 'Woolworths Avonhead', NULL, NULL, NULL, 'NZ', NULL, NULL, 'False'),\n('woolworths:002', 'woolworths', '002', 'woolworths', 'Woolworths Ashburton', NULL, NULL, NULL, 'NZ', NULL, NULL, 'False'),\n('woolworths:003', 'woolworths', '003', 'woolworths', 'Woolworths Northlands', NULL, NULL, NULL, 'NZ', NULL, NULL, 'False'),\n('woolworths:004', 'woolworths', '004', 'woolworths', 'Woolworths Belfast', NULL, NULL, NULL, 'NZ', NULL, NULL, 'False'),\n('woolworths:005', 'woolworths', '005', 'woolworths', 'Woolworths Christchurch Airport', NULL, NULL, NULL, 'NZ', NULL, NULL, 'False'),\n('woolworths:006', 'woolworths', '006', 'woolworths', 'Woolworths Church Corner', NULL, NULL, NULL, 'NZ', NULL, NULL, 'False'),\n('woolworths:007', 'woolworths', '007', 'woolworths', 'Woolworths Colombo St', NULL, NULL, NULL, 'NZ', NULL, NULL, 'False'),\n('woolworths:008', 'woolworths', '008', 'woolworths', 'Woolworths Eastgate', NULL, NULL, NULL, 'NZ', NULL, NULL, 'False'),\n('woolworths:009', 'woolworths', '009', 'woolworths', 'Woolworths Ferrymead', NULL, NULL, NULL, 'NZ', NULL, NULL, 'False'),\n('woolworths:010', 'woolworths', '010', 'woolworths', 'Woolworths Hornby', NULL, NULL, NULL, 'NZ', NULL, NULL, 'False'),\n('woolworths:011', 'woolworths', '011', 'woolworths', 'Woolworths Moorhouse Ave', NULL, NULL, NULL, 'NZ', NULL, NULL, 'False'),\n('woolworths:012', 'woolworths', '012', 'woolworths', 'Woolworths New Brighton', NULL, NULL, NULL, 'NZ', NULL, NULL, 'False'),\n('woolworths:013', 'woolworths', '013', 'woolworths', 'Woolworths The Palms', NULL, NULL, NULL, 'NZ', NULL, NULL, 'False'),\n('newworld:3ee7214b-6df4-4fdb-9dd6-3b2fc252ba6b', 'newworld', '3ee7214b-6df4-4fdb-9dd6-3b2fc252ba6b', 'newworld', NULL, NULL, NULL, NULL, 'NZ', NULL, NULL, 'False'),\n('paknsave:4a279605-eaa8-470d-bcd4-0a9e3c9ab43b', 'paknsave', '4a279605-eaa8-470d-bcd4-0a9e3c9ab43b', 'paknsave', 'PAK\\'nSAVE Riccarton', NULL, NULL, NULL, 'NZ', NULL, NULL, 'False'),\n('paknsave:61dd754e-8525-4b9e-9e08-173389eea8a8', 'paknsave', '61dd754e-8525-4b9e-9e08-173389eea8a8', 'paknsave', 'PAK\\'nSAVE Moorhouse', NULL, NULL, NULL, 'NZ', NULL, NULL, 'False'),\n('paknsave:8cd700ae-d96f-4761-bd7a-805d6b93536d', 'paknsave', '8cd700ae-d96f-4761-bd7a-805d6b93536d', 'paknsave', 'PAK\\'nSAVE Papanui', NULL, NULL, NULL, 'NZ', NULL, NULL, 'False'),\n('paknsave:be4c4780-218e-425a-a90f-63e21773572b', 'paknsave', 'be4c4780-218e-425a-a90f-63e21773572b', 'paknsave', 'PAK\\'nSAVE Hornby', NULL, NULL, NULL, 'NZ', NULL, NULL, 'False'),\n('paknsave:dbca5e00-f7f9-43ae-91de-031ad16f8a92', 'paknsave', 'dbca5e00-f7f9-43ae-91de-031ad16f8a92', 'paknsave', 'PAK\\'nSAVE Wainoni', NULL, NULL, NULL, 'NZ', NULL, NULL, 'False') AS v(store_key, source, store_id, brand_id, store_name, address, city, region, country_code, latitude, longitude, location_verified)) s ON t.store_key=s.store_key WHEN NOT MATCHED THEN INSERT (store_key, source, store_id, brand_id, store_name, address, city, region, country_code, latitude, longitude, location_verified) VALUES (s.store_key, s.source, s.store_id, s.brand_id, s.store_name, s.address, s.city, s.region, s.country_code, s.latitude, s.longitude, s.location_verified)", "MERGE INTO workspace.default.silver_dim_product t USING (SELECT * FROM VALUES ('0001', 'Farmer Brown Eggs Cage-Free Barn Size 6', '1', '12pk', '12', 'pk', 'ea', NULL),\n('0002', 'Farmer Brown Eggs Cage-Free Barn Size 6', '1', '18pk', '18', 'pk', 'ea', NULL),\n('0003', 'Farmer Brown Eggs Cage-Free Barn Size 7', '1', '12pk', '12', 'pk', 'ea', NULL),\n('0004', 'Farmer Brown Eggs Cage-Free Barn Size 7', '1', '18pk', '18', 'pk', 'ea', NULL),\n('0005', 'Farmer Brown Eggs Fresh Colony Size 6', '1', '12pk', '12', 'pk', 'ea', NULL),\n('0006', 'Farmer Brown Eggs Fresh Colony Size 6', '1', '18pk', '18', 'pk', 'ea', NULL),\n('0007', 'Farmer Brown Eggs Fresh Colony Size 7', '1', '12pk', '12', 'pk', 'ea', NULL),\n('0008', 'Farmer Brown Eggs Fresh Colony Size 7', '1', '18pk', '18', 'pk', 'ea', NULL),\n('0009', 'Farmer Brown Eggs Fresh Colony Size 8', '1', '6pk', '6', 'pk', 'ea', NULL),\n('0010', 'Farmer Brown Eggs Fresh Colony Size 8', '1', '10pk', '10', 'pk', 'ea', NULL),\n('0011', 'Woodland Eggs Free Range Size 6 Carton', '1', '10pk', '10', 'pk', 'ea', NULL),\n('0012', 'Woodland Eggs Free Range Size 6 Carton', '1', '18pk', '18', 'pk', 'ea', NULL),\n('0013', 'Woodland Eggs Free Range Size 7 Carton', '1', '6pk', '6', 'pk', 'ea', NULL),\n('0014', 'Woodland Eggs Free Range Size 7 Carton', '1', '10pk', '10', 'pk', 'ea', NULL),\n('0015', 'Henergy Eggs Cage Free Size 6 Carton', '1', '12pk', '12', 'pk', 'ea', NULL),\n('0016', 'Henergy Eggs Cage Free Size 6 Carton', '1', '18pk', '18', 'pk', 'ea', NULL),\n('0017', 'Henergy Eggs Cage Free Size 7 Carton', '1', '12pk', '12', 'pk', 'ea', NULL),\n('0018', 'Henergy Eggs Cage Free Size 7 Carton', '1', '18pk', '18', 'pk', 'ea', NULL),\n('0019', 'Meadow Fresh Original Fresh Milk', '2', '1l', '1', 'l', 'ea', NULL),\n('0020', 'Meadow Fresh Original Fresh Milk', '2', '2l', '2', 'l', 'ea', NULL),\n('0021', 'Meadow Fresh Original Fresh Milk', '2', '3l', '3', 'l', 'ea', NULL),\n('0022', 'Meadow Fresh Calci-Trim Fresh Milk', '2', '1l', '1', 'l', 'ea', NULL),\n('0023', 'Meadow Fresh Calci-Trim Fresh Milk', '2', '2l', '2', 'l', 'ea', NULL),\n('0024', 'Puhoi Valley Fresh Organic Milk', '2', '750ml', '750', 'ml', 'ea', NULL),\n('0025', 'Puhoi Valley Fresh Organic Milk', '2', '1.5l', '1.5', 'l', 'ea', NULL),\n('0026', 'Puhoi Valley Fresh Organic Light Milk', '2', '750ml', '750', 'ml', 'ea', NULL),\n('0027', 'Puhoi Valley Fresh Organic Light Milk', '2', '1.5l', '1.5', 'l', 'ea', NULL),\n('0028', 'Woolworths Milk Standard Bottle', '2', '1l', '1', 'l', 'ea', NULL),\n('0029', 'Woolworths Milk Standard Bottle', '2', '2l', '2', 'l', 'ea', NULL),\n('0030', 'Woolworths Milk Standard Bottle', '2', '3l', '3', 'l', 'ea', NULL),\n('0031', 'Value Standard Milk', '2', '2l', '2', 'l', 'ea', NULL),\n('0032', 'Value Standard Milk', '2', '3l', '3', 'l', 'ea', NULL),\n('0033', 'Value Trim Milk', '2', '2l', '2', 'l', 'ea', NULL),\n('0034', 'Value Trim Milk', '2', '3l', '3', 'l', 'ea', NULL),\n('0035', 'Pams Standard Milk', '2', '1l', '1', 'l', 'ea', NULL),\n('0036', 'Pams Standard Milk', '2', '2l', '2', 'l', 'ea', NULL),\n('0037', 'Pams Standard Milk', '2', '3l', '3', 'l', 'ea', NULL),\n('0038', 'Pams Trim Milk', '2', '1l', '1', 'l', 'ea', NULL),\n('0039', 'Pams Trim Milk', '2', '2l', '2', 'l', 'ea', NULL),\n('0040', 'Pams Trim Milk', '2', '3l', '3', 'l', 'ea', NULL),\n('0041', 'Beef Fillet Steak', '3', 'kg', NULL, NULL, 'kg', NULL),\n('0042', 'Beef Rump Steak', '3', 'kg', NULL, NULL, 'kg', NULL),\n('0043', 'NZ Beef Chuck Steak', '3', 'kg', NULL, NULL, 'kg', NULL),\n('0044', 'NZ Beef T Bone Steak', '3', 'kg', NULL, NULL, 'kg', NULL),\n('0045', 'Prime Beef Mince', '3', 'kg', NULL, NULL, 'kg', NULL),\n('0046', 'NZ Premium Beef Mince', '3', 'kg', NULL, NULL, 'kg', NULL),\n('0047', 'NZ Diced Beef', '3', 'kg', NULL, NULL, 'kg', NULL),\n('0048', 'Lamb Leg', '3', 'kg', NULL, NULL, 'kg', NULL),\n('0049', 'Lamb Shoulder Chops', '3', 'kg', NULL, NULL, 'kg', NULL),\n('0050', 'Lamb Mince', '3', 'kg', NULL, NULL, 'kg', NULL),\n('0051', 'Diced Lamb', '3', 'kg', NULL, NULL, 'kg', NULL) AS v(product_id, product_name, category_id, specification, quantity, quantity_unit, comparison_unit, product_brand_name)) s ON t.product_id=s.product_id WHEN NOT MATCHED THEN INSERT (product_id, product_name, category_id, specification, quantity, quantity_unit, comparison_unit, product_brand_name) VALUES (s.product_id, s.product_name, s.category_id, s.specification, s.quantity, s.quantity_unit, s.comparison_unit, s.product_brand_name)", "MERGE INTO workspace.default.silver_dim_product_alias t USING (SELECT * FROM VALUES ('newworld', 'Farmer Brown Cage-Free Barn Size 6 Eggs', '12pk', '0001'),\n('paknsave', 'Farmer Brown Cage-Free Barn Size 6 Eggs', '12pk', '0001'),\n('woolworths', 'Farmer Brown Eggs Cage Free Barn Size 6 Carton 12pack', '12pk', '0001'),\n('newworld', 'Farmer Brown Cage-Free Barn Size 6 Eggs', '18pk', '0002'),\n('paknsave', 'Farmer Brown Cage-Free Barn Size 6 Eggs', '18pk', '0002'),\n('woolworths', 'Farmer Brown Eggs Cage Free Barn Size 6 Carton 18pack', '18pk', '0002'),\n('newworld', 'Farmer Brown Cage-Free Barn Size 7 Eggs', '12pk', '0003'),\n('paknsave', 'Farmer Brown Cage-Free Barn Size 7 Eggs', '12pk', '0003'),\n('woolworths', 'Farmer Brown Eggs Cage Free Barn Size 7 Carton 12pack', '12pk', '0003'),\n('newworld', 'Farmer Brown Cage-Free Barn Size 7 Eggs', '18pk', '0004'),\n('paknsave', 'Farmer Brown Cage-Free Barn Size 7 Eggs', '18pk', '0004'),\n('woolworths', 'Farmer Brown Eggs Cage Free Barn Size 7 Carton 18pack', '18pk', '0004'),\n('newworld', 'Farmer Brown Fresh Colony Size 6 Eggs', '12pk', '0005'),\n('paknsave', 'Farmer Brown Fresh Colony Size 6 Eggs', '12pk', '0005'),\n('woolworths', 'Farmer Brown Eggs Colony Size 6 Carton 12pack', '12pk', '0005'),\n('newworld', 'Farmer Brown Fresh Colony Size 6 Eggs', '18pk', '0006'),\n('paknsave', 'Farmer Brown Fresh Colony Size 6 Eggs', '18pk', '0006'),\n('woolworths', 'Farmer Brown Eggs Colony Size 6 Carton 18pack', '18pk', '0006'),\n('newworld', 'Farmer Brown Fresh Colony Size 7 Eggs', '12pk', '0007'),\n('paknsave', 'Farmer Brown Fresh Colony Size 7 Eggs', '12pk', '0007'),\n('woolworths', 'Farmer Brown Eggs Colony Size 7 Carton 12pack', '12pk', '0007'),\n('newworld', 'Farmer Brown Fresh Colony Size 7 Eggs', '18pk', '0008'),\n('paknsave', 'Farmer Brown Fresh Colony Size 7 Eggs', '18pk', '0008'),\n('woolworths', 'Farmer Brown Eggs Colony Size 7 Carton 18pack', '18pk', '0008'),\n('newworld', 'Farmer Brown Fresh Colony Size 8 Eggs', '6pk', '0009'),\n('paknsave', 'Farmer Brown Fresh Colony Size 8 Eggs', '6pk', '0009'),\n('woolworths', 'Farmer Brown Eggs Colony Size 8 Carton 6pack', '6pk', '0009'),\n('newworld', 'Farmer Brown Fresh Colony Size 8 Eggs', '10pk', '0010'),\n('paknsave', 'Farmer Brown Fresh Colony Size 8 Eggs', '10pk', '0010'),\n('woolworths', 'Farmer Brown Eggs Colony Size 8 Carton 10pack', '10pk', '0010'),\n('newworld', 'Woodland Free Range Size 6 Eggs', '10pk', '0011'),\n('paknsave', 'Woodland Free Range Size 6 Eggs', '10pk', '0011'),\n('woolworths', 'Woodland Eggs Free Range Size 6 Carton 10pack', '10pk', '0011'),\n('newworld', 'Woodland Free Range Size 6 Eggs', '18pk', '0012'),\n('paknsave', 'Woodland Free Range Size 6 Eggs', '18pk', '0012'),\n('woolworths', 'Woodland Eggs Free Range Size 6 Carton 18pack', '18pk', '0012'),\n('newworld', 'Woodland Free Range Size 7 Eggs', '6pk', '0013'),\n('paknsave', 'Woodland Free Range Size 7 Eggs', '6pk', '0013'),\n('woolworths', 'Woodland Eggs Free Range Size 7 Carton 6pack', '6pk', '0013'),\n('newworld', 'Woodland Free Range Size 7 Eggs', '10pk', '0014'),\n('paknsave', 'Woodland Free Range Size 7 Eggs', '10pk', '0014'),\n('woolworths', 'Woodland Eggs Free Range Size 7 Carton 10pack', '10pk', '0014'),\n('newworld', 'Henergy Cage-Free Size 6 Eggs', '12pk', '0015'),\n('paknsave', 'Henergy Cage-Free Size 6 Eggs', '12pk', '0015'),\n('woolworths', 'Henergy Eggs Cage Free Size 6 Carton 12pack', '12pk', '0015'),\n('newworld', 'Henergy Cage-Free Size 6 Eggs', '18pk', '0016'),\n('paknsave', 'Henergy Cage-Free Size 6 Eggs', '18pk', '0016'),\n('woolworths', 'Henergy Eggs Cage Free Size 6 Carton 18pack', '18pk', '0016'),\n('newworld', 'Henergy Cage-Free Size 7 Eggs', '12pk', '0017'),\n('paknsave', 'Henergy Cage-Free Size 7 Eggs', '12pk', '0017'),\n('woolworths', 'Henergy Eggs Cage Free Size 7 Carton 12pack', '12pk', '0017'),\n('newworld', 'Henergy Cage-Free Size 7 Eggs', '18pk', '0018'),\n('paknsave', 'Henergy Cage-Free Size 7 Eggs', '18pk', '0018'),\n('woolworths', 'Henergy Eggs Cage Free Size 7 Carton 18pack', '18pk', '0018'),\n('newworld', 'Meadow Fresh Original Fresh Milk', '1l', '0019'),\n('paknsave', 'Meadow Fresh Original Fresh Milk', '1l', '0019'),\n('woolworths', 'Meadow Fresh Milk Standard Original Bottle 1L', '1l', '0019'),\n('newworld', 'Meadow Fresh Original Fresh Milk', '2l', '0020'),\n('paknsave', 'Meadow Fresh Original Fresh Milk', '2l', '0020'),\n('woolworths', 'Meadow Fresh Milk Standard Original Bottle 2L', '2l', '0020'),\n('newworld', 'Meadow Fresh Original Fresh Milk', '3l', '0021'),\n('paknsave', 'Meadow Fresh Original Fresh Milk', '3l', '0021'),\n('woolworths', 'Meadow Fresh Milk Standard Original Bottle 3L', '3l', '0021'),\n('newworld', 'Meadow Fresh Calci-Trim Fresh Milk', '1l', '0022'),\n('paknsave', 'Meadow Fresh Calci-Trim Fresh Milk', '1l', '0022'),\n('woolworths', 'Meadow Fresh Milk Trim Calci-Trim Bottle 1L', '1l', '0022'),\n('newworld', 'Meadow Fresh Calci-Trim Fresh Milk', '2l', '0023'),\n('paknsave', 'Meadow Fresh Calci-Trim Fresh Milk', '2l', '0023'),\n('woolworths', 'Meadow Fresh Milk Trim Calci-Trim Bottle 2L', '2l', '0023'),\n('newworld', 'Puhoi Valley Fresh Organic Milk', '750ml', '0024'),\n('paknsave', 'Puhoi Valley Fresh Organic Milk', '750ml', '0024'),\n('woolworths', 'Puhoi Valley Milk Organic Homogenised Bottle 750mL', '750ml', '0024'),\n('newworld', 'Puhoi Valley Fresh Organic Milk', '1.5l', '0025'),\n('paknsave', 'Puhoi Valley Fresh Organic Milk', '1.5l', '0025'),\n('woolworths', 'Puhoi Valley Milk Organic Homogenised Bottle 1.5L', '1.5l', '0025'),\n('newworld', 'Puhoi Valley Fresh Organic Light Milk', '750ml', '0026'),\n('paknsave', 'Puhoi Valley Fresh Organic Light Milk', '750ml', '0026'),\n('woolworths', 'Puhoi organic Milk LightBottle 750mL', '750ml', '0026'),\n('newworld', 'Puhoi Valley Fresh Organic Light Milk', '1.5l', '0027'),\n('paknsave', 'Puhoi Valley Fresh Organic Light Milk', '1.5l', '0027'),\n('woolworths', 'Puhoi organic Milk LightBottle 1.5L', '1.5l', '0027'),\n('woolworths', 'Woolworths Milk Standard Bottle 1L', '1l', '0028'),\n('woolworths', 'Woolworths Milk Standard Bottle 2L', '2l', '0029'),\n('woolworths', 'Woolworths Milk Standard Bottle 3L', '3l', '0030'),\n('newworld', 'Value Standard Milk', '2l', '0031'),\n('paknsave', 'Value Standard Milk', '2l', '0031'),\n('newworld', 'Value Standard Milk', '3l', '0032'),\n('paknsave', 'Value Standard Milk', '3l', '0032'),\n('newworld', 'Value Trim Milk', '2l', '0033'),\n('paknsave', 'Value Trim Milk', '2l', '0033'),\n('newworld', 'Value Trim Milk', '3l', '0034'),\n('paknsave', 'Value Trim Milk', '3l', '0034'),\n('newworld', 'Pams Standard Milk', '1l', '0035'),\n('paknsave', 'Pams Standard Milk', '1l', '0035'),\n('newworld', 'Pams Standard Milk', '2l', '0036'),\n('paknsave', 'Pams Standard Milk', '2l', '0036'),\n('newworld', 'Pams Standard Milk', '3l', '0037'),\n('paknsave', 'Pams Standard Milk', '3l', '0037'),\n('newworld', 'Pams Trim Milk', '1l', '0038'),\n('paknsave', 'Pams Trim Milk', '1l', '0038'),\n('newworld', 'Pams Trim Milk', '2l', '0039'),\n('paknsave', 'Pams Trim Milk', '2l', '0039'),\n('newworld', 'Pams Trim Milk', '3l', '0040'),\n('paknsave', 'Pams Trim Milk', '3l', '0040'),\n('newworld', 'Beef Fillet Steak', 'kg', '0041'),\n('paknsave', 'Beef Fillet Steak', 'kg', '0041'),\n('woolworths', 'Woolworths NZ Beef Eye Fillet Steak Whole Min Order 2kg', 'kg', '0041'),\n('newworld', 'Beef Rump Steak', 'kg', '0042'),\n('paknsave', 'Beef Rump Steak', 'kg', '0042'),\n('woolworths', 'Woolworths NZ Beef Rump Steak Grass Fed Min Order 500g', 'kg', '0042'),\n('newworld', 'NZ Beef Chuck Steak', 'kg', '0043'),\n('paknsave', 'NZ Beef Chuck Steak', 'kg', '0043'),\n('woolworths', 'Woolworths NZ Beef Chuck Steak Grass Fed Medium Tray Min Order 700g', 'kg', '0043'),\n('newworld', 'NZ Beef T Bone Steak', 'kg', '0044'),\n('paknsave', 'NZ Beef T Bone Steak', 'kg', '0044'),\n('woolworths', 'Woolworths NZ Beef T-Bone Steak Grass Fed Medium Tray Min Order 700g', 'kg', '0044'),\n('newworld', 'Prime Beef Mince', 'kg', '0045'),\n('paknsave', 'Prime Beef Mince', 'kg', '0045'),\n('woolworths', 'Woolworths Beef Mince Prime Min Order 500g', 'kg', '0045'),\n('newworld', 'NZ Premium Beef Mince', 'kg', '0046'),\n('paknsave', 'NZ Premium Beef Mince', 'kg', '0046'),\n('woolworths', 'Woolworths Beef Mince Premium Min Order 500g', 'kg', '0046'),\n('newworld', 'NZ Diced Beef', 'kg', '0047'),\n('paknsave', 'NZ Diced Beef', 'kg', '0047'),\n('woolworths', 'Woolworths NZ Beef Diced Premium Grass Fed Min Order 650g', 'kg', '0047'),\n('newworld', 'Lamb Leg Roast', 'kg', '0048'),\n('paknsave', 'Lamb Leg Roast', 'kg', '0048'),\n('woolworths', 'Woolworths Lamb Roast Leg', 'kg', '0048'),\n('newworld', 'Lamb Shoulder Chops', 'kg', '0049'),\n('paknsave', 'Lamb Shoulder Chops', 'kg', '0049'),\n('woolworths', 'Woolworths Lamb Chops Shoulder Bbq 2 Piece Min Order 500g', 'kg', '0049'),\n('newworld', 'Lamb Mince', 'kg', '0050'),\n('paknsave', 'Lamb Mince', 'kg', '0050'),\n('woolworths', 'Woolworths Lamb Mince Prime Medium Tray Min Order 400g', 'kg', '0050'),\n('newworld', 'Diced Lamb', 'kg', '0051'),\n('paknsave', 'Diced Lamb', 'kg', '0051'),\n('woolworths', 'Woolworths Lamb Diced Small Tray Min Order 500g', 'kg', '0051') AS v(source, original_name, original_specification, product_id)) s ON t.source=s.source AND t.original_name=s.original_name AND t.original_specification=s.original_specification WHEN NOT MATCHED THEN INSERT (source, original_name, original_specification, product_id) VALUES (s.source, s.original_name, s.original_specification, s.product_id)"]
for statement in statements:
    spark.sql(statement)
print("模型及参考数据初始化完成")


## 门店地址与坐标
只补缺失坐标/来源，保留已有核实结果和手工维护的资料。供应商坐标未完成地图核对时 location_verified=false。


In [ ]:
location_sql = "MERGE INTO workspace.default.silver_dim_store t USING (SELECT * FROM VALUES ('woolworths:001','Woolworths Avonhead','Cnr Withells Road & Merrin St, Avonhead, Christchurch, Canterbury, New Zealand','Christchurch','Canterbury','NZ',-43.510368,172.556909,false,'https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','2026-10-26','supplier_directory_pending_map_check',true,'woolworths','001','woolworths'),\n('woolworths:002','Woolworths Ashburton','Cnr East & Peter Streets, Ashburton, Canterbury, New Zealand','Ashburton','Canterbury','NZ',-43.901624,171.752552,false,'https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','2026-10-26','supplier_directory_pending_map_check',false,'woolworths','002','woolworths'),\n('woolworths:003','Woolworths Northlands','85 Main North Road, Papanui, Christchurch, Canterbury, New Zealand','Christchurch','Canterbury','NZ',-43.492064,172.611694,false,'https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','2026-10-26','supplier_directory_pending_map_check',true,'woolworths','003','woolworths'),\n('woolworths:004','Woolworths Belfast','755 Main North Road, Belfast, Christchurch, Canterbury 8051, New Zealand','Christchurch','Canterbury','NZ',-43.44776997837326,172.62849342372328,false,'https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','2026-10-26','supplier_directory_pending_map_check',true,'woolworths','004','woolworths'),\n('woolworths:005','Woolworths Christchurch Airport','530 - 546A Memorial Avenue, Christchurch, Canterbury 8053, New Zealand','Christchurch','Canterbury','NZ',-43.48986,172.54771,true,'https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','https://mapcarta.com/W540962413','2026-10-26','directory_and_osm_cross_checked',true,'woolworths','005','woolworths'),\n('woolworths:006','Woolworths Church Corner','Cnr Riccarton Road & Hansons Lane, Upper Riccarton, Christchurch, Canterbury, New Zealand','Christchurch','Canterbury','NZ',-43.53276,172.57308,true,'https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','https://mapcarta.com/W34369592','2026-10-26','directory_and_osm_cross_checked',true,'woolworths','006','woolworths'),\n('woolworths:007','Woolworths Colombo St','219 Colombo Street, Christchurch, Canterbury, New Zealand','Christchurch','Canterbury','NZ',-43.55333,172.6356,true,'https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','https://mapcarta.com/W53916748','2026-10-26','directory_and_osm_cross_checked',true,'woolworths','007','woolworths'),\n('woolworths:008','Woolworths Eastgate','Cnr Linwood Avenue & Buckleys Roads, Linwood, Christchurch, Canterbury, New Zealand','Christchurch','Canterbury','NZ',-43.533266,172.673275,false,'https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','2026-10-26','supplier_directory_pending_map_check',true,'woolworths','008','woolworths'),\n('woolworths:009','Woolworths Ferrymead','999 Ferry Rd, Ferrymead, Christchurch, Canterbury 8023, New Zealand','Christchurch','Canterbury','NZ',-43.556637818004944,172.70229836384624,false,'https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','2026-10-26','supplier_directory_pending_map_check',true,'woolworths','009','woolworths'),\n('woolworths:010','Woolworths Hornby','17 Chappie Place, Hornby, Christchurch, Canterbury, New Zealand','Christchurch','Canterbury','NZ',-43.542514,172.527313,false,'https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','2026-10-26','supplier_directory_pending_map_check',true,'woolworths','010','woolworths'),\n('woolworths:011','Woolworths Moorhouse Ave','Cnr Moorhouse Avenue & Madras Street, Christchurch, Canterbury, New Zealand','Christchurch','Canterbury','NZ',-43.53877,172.64177,true,'https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','https://mapcarta.com/W101515749','2026-10-26','directory_and_osm_cross_checked',true,'woolworths','011','woolworths'),\n('woolworths:012','Woolworths New Brighton','12 Hawke Street, New Brighton, Christchurch, Canterbury, New Zealand','Christchurch','Canterbury','NZ',-43.506754,172.729847,false,'https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','2026-10-26','supplier_directory_pending_map_check',true,'woolworths','012','woolworths'),\n('woolworths:013','Woolworths The Palms','Marshlands Road, Shirley, Christchurch, Canterbury, New Zealand','Christchurch','Canterbury','NZ',-43.507661,172.663476,false,'https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','https://silverfernfarms.com/nz/en/contact-us/where-to-buy/partner-details','2026-10-26','supplier_directory_pending_map_check',true,'woolworths','013','woolworths'),\n('paknsave:be4c4780-218e-425a-a90f-63e21773572b','PAK\\'nSAVE Hornby','The Hub 418 Main S Rd, Hornby, Christchurch 8042','Christchurch','Canterbury','NZ',-43.542540854010916,172.52275767116396,true,'https://www.paknsave.co.nz/south-island/canterbury/hornby','https://www.paknsave.co.nz/south-island/canterbury/hornby','2026-10-26','official_directory',true,'paknsave','be4c4780-218e-425a-a90f-63e21773572b','paknsave'),\n('paknsave:61dd754e-8525-4b9e-9e08-173389eea8a8','PAK\\'nSAVE Moorhouse','297 Moorhouse Ave, Sydenham, Christchurch 8011','Christchurch','Canterbury','NZ',-43.53889986394847,172.63831266242676,true,'https://www.paknsave.co.nz/south-island/canterbury/moorhouse','https://www.paknsave.co.nz/south-island/canterbury/moorhouse','2026-10-26','official_directory',true,'paknsave','61dd754e-8525-4b9e-9e08-173389eea8a8','paknsave'),\n('paknsave:8cd700ae-d96f-4761-bd7a-805d6b93536d','PAK\\'nSAVE Papanui','171 Main North Road, Papanui, Christchurch, New Zealand','Christchurch','Canterbury','NZ',-43.4853687809161,172.61498059995094,true,'https://www.paknsave.co.nz/south-island/canterbury/papanui','https://www.paknsave.co.nz/south-island/canterbury/papanui','2026-10-26','official_directory',true,'paknsave','8cd700ae-d96f-4761-bd7a-805d6b93536d','paknsave'),\n('paknsave:4a279605-eaa8-470d-bcd4-0a9e3c9ab43b','PAK\\'nSAVE Riccarton','Westfield Riccarton, 129 Riccarton Rd, Riccarton, Christchurch 8041','Christchurch','Canterbury','NZ',-43.53099691110821,172.59617139601744,true,'https://www.paknsave.co.nz/south-island/canterbury/riccarton','https://www.paknsave.co.nz/south-island/canterbury/riccarton','2026-10-26','official_directory',true,'paknsave','4a279605-eaa8-470d-bcd4-0a9e3c9ab43b','paknsave'),\n('paknsave:dbca5e00-f7f9-43ae-91de-031ad16f8a92','PAK\\'nSAVE Wainoni','174 Wainoni Rd, Avondale, Christchurch 8061','Christchurch','Canterbury','NZ',-43.513245167402054,172.69400674498297,true,'https://www.paknsave.co.nz/south-island/canterbury/wainoni','https://www.paknsave.co.nz/south-island/canterbury/wainoni','2026-10-26','official_directory',true,'paknsave','dbca5e00-f7f9-43ae-91de-031ad16f8a92','paknsave'),\n('newworld:b3158cd8-72b9-40e5-8c4e-abd43c1be305','New World Bishopdale','Bishopdale Mall, Cnr Farrington Ave & Harewood Rd, Christchurch 8053','Christchurch','Canterbury','NZ',-43.4883881678695,172.58712833013726,true,'https://www.newworld.co.nz/south-island/canterbury/bishopdale','https://www.newworld.co.nz/south-island/canterbury/bishopdale','2026-10-26','official_directory',true,'newworld','b3158cd8-72b9-40e5-8c4e-abd43c1be305','newworld'),\n('newworld:c1aaac72-38c0-4cc0-ad05-f241047d88c5','New World Durham Street','175 Durham Street South, Christchurch 8011','Christchurch','Canterbury','NZ',-43.538668,172.633512,true,'https://www.newworld.co.nz/south-island/canterbury/durham-street','https://www.newworld.co.nz/south-island/canterbury/durham-street','2026-10-26','official_directory',true,'newworld','c1aaac72-38c0-4cc0-ad05-f241047d88c5','newworld'),\n('newworld:81f807e9-1879-484c-b27e-ded56245c6a4','New World Fendalton','19-23 Memorial Avenue, Christchurch 8053','Christchurch','Canterbury','NZ',-43.51788708337433,172.58833822698364,true,'https://www.newworld.co.nz/south-island/canterbury/fendalton','https://www.newworld.co.nz/south-island/canterbury/fendalton','2026-10-26','official_directory',true,'newworld','81f807e9-1879-484c-b27e-ded56245c6a4','newworld'),\n('newworld:fc91d59f-6ab5-4447-8737-125e09e8e50e','New World Ferry Road','7-11 St John Street, Woolston, Christchurch 8062','Christchurch','Canterbury','NZ',-43.54936274246441,172.68439926745907,true,'https://www.newworld.co.nz/south-island/canterbury/ferry-road','https://www.newworld.co.nz/south-island/canterbury/ferry-road','2026-10-26','official_directory',true,'newworld','fc91d59f-6ab5-4447-8737-125e09e8e50e','newworld'),\n('newworld:95d161ea-9a31-4fca-acbe-96f271d627df','New World Halswell','346 Halswell Road, Christchurch 8025','Christchurch','Canterbury','NZ',-43.58062650304889,172.56601248941047,true,'https://www.newworld.co.nz/south-island/canterbury/halswell','https://www.newworld.co.nz/south-island/canterbury/halswell','2026-10-26','official_directory',true,'newworld','95d161ea-9a31-4fca-acbe-96f271d627df','newworld'),\n('newworld:c6abac35-b75f-4a02-9b43-7ad5a7c7aa37','New World Ilam','47C - 57C Peer St, Upper Riccarton, Christchurch 8041','Christchurch','Canterbury','NZ',-43.52601457999897,172.5701913994751,true,'https://www.newworld.co.nz/south-island/canterbury/ilam','https://www.newworld.co.nz/south-island/canterbury/ilam','2026-10-26','official_directory',true,'newworld','c6abac35-b75f-4a02-9b43-7ad5a7c7aa37','newworld'),\n('newworld:0637c08c-ac36-4940-8582-38e44a4812fa','New World Northwood','2 Mounter Ave, Christchurch 8051','Christchurch','Canterbury','NZ',-43.4596218,172.62246570000002,true,'https://www.newworld.co.nz/south-island/canterbury/northwood','https://www.newworld.co.nz/south-island/canterbury/northwood','2026-10-26','official_directory',true,'newworld','0637c08c-ac36-4940-8582-38e44a4812fa','newworld'),\n('newworld:5b8f8e3b-e1a0-4a11-b16b-9cfe782c124e','New World Prestons','420 Marshland Rd, Marshland, Christchurch 8083','Christchurch','Canterbury','NZ',-43.4751212,172.6598818,true,'https://www.newworld.co.nz/south-island/canterbury/prestons','https://www.newworld.co.nz/south-island/canterbury/prestons','2026-10-26','official_directory',true,'newworld','5b8f8e3b-e1a0-4a11-b16b-9cfe782c124e','newworld'),\n('newworld:bae1da58-ac6b-4ef2-816d-4932691218d1','New World St Martins','92 Wilsons Road, Christchurch 8023','Christchurch','Canterbury','NZ',-43.55767693822415,172.6543260490738,true,'https://www.newworld.co.nz/south-island/canterbury/st-martins','https://www.newworld.co.nz/south-island/canterbury/st-martins','2026-10-26','official_directory',true,'newworld','bae1da58-ac6b-4ef2-816d-4932691218d1','newworld'),\n('newworld:a0d86b5f-fdf4-44d1-b7a2-f418bdb37f4e','New World Stanmore','288 Stanmore Road, Christchurch 8013','Christchurch','Canterbury','NZ',-43.5199514,172.65720199999998,true,'https://www.newworld.co.nz/south-island/canterbury/stanmore','https://www.newworld.co.nz/south-island/canterbury/stanmore','2026-10-26','official_directory',true,'newworld','a0d86b5f-fdf4-44d1-b7a2-f418bdb37f4e','newworld'),\n('newworld:3ee7214b-6df4-4fdb-9dd6-3b2fc252ba6b','New World Wigram','51 Skyhawk Road, Wigram 8042 Christchurch, New Zealand','Christchurch','Canterbury','NZ',-43.55264520254251,172.55784152327874,true,'https://www.newworld.co.nz/south-island/canterbury/wigram','https://www.newworld.co.nz/south-island/canterbury/wigram','2026-10-26','official_directory',true,'newworld','3ee7214b-6df4-4fdb-9dd6-3b2fc252ba6b','newworld') AS v(store_key,store_name,address,city,region,country_code,latitude,longitude,location_verified,location_source_url,coordinate_source_url,location_checked_date,location_status,is_christchurch_urban,source,store_id,brand_id)) s\nON t.store_key=s.store_key\nWHEN MATCHED AND (t.latitude IS NULL OR t.longitude IS NULL OR t.coordinate_source_url IS NULL)\nTHEN UPDATE SET t.store_name=s.store_name, t.address=s.address, t.city=s.city, t.region=s.region, t.country_code=s.country_code, t.latitude=s.latitude, t.longitude=s.longitude, t.location_verified=s.location_verified, t.location_source_url=s.location_source_url, t.coordinate_source_url=s.coordinate_source_url, t.location_checked_date=s.location_checked_date, t.location_status=s.location_status, t.is_christchurch_urban=s.is_christchurch_urban, t.source=s.source, t.store_id=s.store_id, t.brand_id=s.brand_id\nWHEN NOT MATCHED THEN INSERT (store_key,store_name,address,city,region,country_code,latitude,longitude,location_verified,location_source_url,coordinate_source_url,location_checked_date,location_status,is_christchurch_urban,source,store_id,brand_id) VALUES (s.store_key,s.store_name,s.address,s.city,s.region,s.country_code,s.latitude,s.longitude,s.location_verified,s.location_source_url,s.coordinate_source_url,s.location_checked_date,s.location_status,s.is_christchurch_urban,s.source,s.store_id,s.brand_id)"
spark.sql(location_sql)
